# **Final Project**

### Introduction

This project extends the recommendation system developed in Project 5 by implementing a hybrid recommender that combines collaborative filtering with content-based filtering. The existing Alternating Least Squares (ALS) model continues to generate personalized movie recommendations based on user rating patterns, while movie director information is incorporated to refine the final recommendation rankings. By integrating collaborative filtering with director metadata, the hybrid system aims to improve recommendation quality beyond what can be achieved using user–item interactions alone. The performance of the proposed model is evaluated by comparing it with the baseline ALS recommender using both prediction accuracy and recommendation quality metrics. In addition to RMSE and MAE, recommendation quality is assessed using Precision@10, Recall@10, novelty, and diversity.


### Methodology

The hybrid recommendation system extends the baseline Alternating Least Squares (ALS) collaborative filtering model developed in Project 5 by incorporating movie director metadata into the recommendation process. The ALS model is first trained using the MovieLens 1M dataset to learn latent user and movie features from historical rating data and generate candidate movie recommendations based on predicted ratings.

Movie director information is then integrated into the recommendation pipeline to incorporate content-based information alongside collaborative filtering predictions. User director preference profiles are constructed from positively rated movies, allowing director metadata to contribute to the refinement of the recommendation rankings.

The performance of the hybrid recommender is evaluated by comparing it with the baseline ALS model. RMSE and MAE are used to evaluate rating prediction accuracy, while Precision@10, Recall@10, novelty, and diversity are used to assess the quality of the generated recommendation lists. The hybrid model is considered successful if it improves the relevance, novelty, or diversity of the recommendation lists while maintaining competitive overall performance relative to the baseline ALS recommender.


### Data Preparation, Exploration, and Preprocessing

This project uses the same preprocessed MovieLens 1M dataset developed in Project 5. Since the dataset had already been cleaned and prepared for collaborative filtering, extensive preprocessing was not required. However, the dataset was re-examined to verify its quality before training the hybrid recommender. This included checking for missing values, confirming the number of users, movies, and ratings, and reviewing the distribution of user and movie rating counts to ensure the data remained suitable for model training. The movie metadata containing the ***directedBy*** attribute was validated and linked with the movie ratings so that director information could be incorporated into the hybrid recommendation process.



In [ ]:
import pandas as pandas


from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructField, FloatType, StringType, IntegerType, StructType
from pyspark import StorageLevel
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast
from IPython.display import display

spark = SparkSession.builder.master("local").appName("test").getOrCreate()


In [ ]:
# Stop the Spark session, for debugging
# spark.catalog.clearCache()
# spark.stop()

In [7]:
movie_ratings = spark.read.csv(
    "/Users/sandy/Desktop/DATA 612/project 7/movie_ratings_cleaned.csv", 
    header=True, inferSchema=True)

movie_ratings.printSchema()
movie_ratings.show()

root
 |-- movieId: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- directedBy: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+------+------+--------------------+------------------+--------------------+
|movieId|userId|rating|               title|        directedBy|              genres|
+-------+------+------+--------------------+------------------+--------------------+
|  91658|142882|   2.5|Girl with the Dra...|     David Fincher|      Drama,Thriller|
|   4344|142882|   1.0|    Swordfish (2001)|      Dominic Sena|  Action,Crime,Drama|
|  45720|142882|   2.0|Devil Wears Prada...|     David Frankel|        Comedy,Drama|
|   4734|142882|   2.0|Jay and Silent Bo...|       Kevin Smith|    Adventure,Comedy|
|  91542|142882|   2.0|Sherlock Holmes: ...|       Guy Ritchie|Action,Adventure,...|
|   4446|142882|   1.5|Final Fantasy: Th...|Hironobu Sakaguchi|Adventure,Animati

In [8]:
movie_ratings.select(
    [
        F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
        for c in movie_ratings.columns
    ]
).show()

+-------+------+------+-----+----------+------+
|movieId|userId|rating|title|directedBy|genres|
+-------+------+------+-----+----------+------+
|      0|     0|     0|    0|         0|     0|
+-------+------+------+-----+----------+------+



In [9]:
print("User rating counts:")
(
    movie_ratings.groupBy("userId")
    .count()
    .describe()
    .show()
)

print("Movie rating counts:")
(
    movie_ratings.groupBy("movieId")
    .count()
    .describe()
    .show()
)

User rating counts:


+-------+------------------+-----------------+
|summary|            userId|            count|
+-------+------------------+-----------------+
|  count|            104661|           104661|
|   mean|153617.99270024174|95.52603166413468|
| stddev|31211.062127929865|214.4697461807079|
|    min|            100000|                1|
|    max|            206985|            19764|
+-------+------------------+-----------------+

Movie rating counts:


+-------+-----------------+------------------+
|summary|          movieId|             count|
+-------+-----------------+------------------+
|  count|            49151|             49151|
|   mean|100619.5436715428|203.41091737706253|
| stddev|58690.67033616423|1163.0664075609682|
|    min|                1|                 1|
|    max|           183335|             42120|
+-------+-----------------+------------------+



### Data Split
To ensure a fair comparison between the baseline ALS model and the proposed hybrid recommender, the same data split from Project 5 was retained. The dataset was first split into training (80%) and testing (20%) on a per-user basis, ensuring that each user remained represented in the training set while reserving a portion of their ratings for testing whenever possible. This approach allows the model to learn each user's preferences while evaluating its ability to predict ratings that were not observed during training.

The training set was then further divided into a final training set (90%) and a validation set (10%) using the same user-based strategy, again ensuring that users with only a single rating remained in the training data. The validation set was used for model selection and hyperparameter tuning, while the independent test set provided an unbiased estimate of the model's final performance.

Although this splitting strategy may result in some movies being absent from the final training set, these movies were intentionally not forced back into the training data. Retaining them only in the validation or test sets preserves the realism of the evaluation by allowing the model to encounter unseen or sparsely rated movies, which better reflects real-world recommendation scenarios where new or less-popular content may appear. This approach also avoids artificially inflating the model's performance by ensuring that evaluation is conducted on genuinely unseen data. Using the same data partitions as Project 5 also ensures that any differences in performance can be attributed to the addition of the hybrid recommendation approach rather than changes in the underlying training and testing data.


In [10]:
# ==========================================================
# STEP 1: Split into Train (80%) and Test (20%)
# ==========================================================

# Create a random ordering of each user's ratings
user_window = Window.partitionBy("userId").orderBy(F.rand(seed=52))

# Number each rating for every user
# Also count how many ratings each user has
df = (
    movie_ratings
    .withColumn("row_num", F.row_number().over(user_window))
    .withColumn("num_ratings", F.count("*").over(Window.partitionBy("userId")))
)

# Compute how many ratings should be placed in the test set
# Hold out roughly 20% of each user's ratings
# BUT keep users with only 1 rating entirely in training
df = df.withColumn(
    "test_size",
    F.when(
        F.col("num_ratings") == 1,
        F.lit(0)  # users with one rating get no test samples
    ).otherwise(
        F.greatest(
            F.lit(1),
            (F.col("num_ratings") * 0.2).cast("int")
        )
    )
)

# First "test_size" ratings become test set
test = df.filter(F.col("row_num") <= F.col("test_size"))

# Remaining ratings become training set
train = df.filter(F.col("row_num") > F.col("test_size"))

In [11]:
# ==========================================================
# STEP 2: Split Train into Final Train (90%) and Validation (10%)
# ==========================================================

# Randomize the ratings again inside each user
train_window = Window.partitionBy("userId").orderBy(F.rand(seed=52))

# Number each rating for every user
train = (
    train
    .withColumn("row_num", F.row_number().over(train_window))
    .withColumn("num_ratings", F.count("*").over(Window.partitionBy("userId")))
)

# Hold out 10% for validation
# Keep single-rating users in training
train = train.withColumn(
    "val_size",
    F.when(
        F.col("num_ratings") == 1,
        F.lit(0)
    ).otherwise(
        F.greatest(
            F.lit(1),
            (F.col("num_ratings") * 0.1).cast("int")
        )
    )
)

# First val_size ratings become validation
val = train.filter(F.col("row_num") <= F.col("val_size"))

# Remaining ratings become the final training set
train_final = train.filter(F.col("row_num") > F.col("val_size"))

In [12]:
helper_columns = [
    "row_num",
    "num_ratings",
    "test_size",
    "val_size"
]

train_final = train_final.drop(*[c for c in helper_columns if c in train_final.columns])
val = val.drop(*[c for c in helper_columns if c in val.columns])
test = test.drop(*[c for c in helper_columns if c in test.columns])

In [13]:
# ==========================================================
# Check dataset sizes
# ==========================================================

train_count = train_final.count()
val_count = val.count()
test_count = test.count()

print("Train:", train_count)
print("Validation:", val_count)
print("Test:", test_count)

print("Total:", train_count + val_count + test_count)
print("Expected:", movie_ratings.count())

Train: 7254551
Validation: 774499
Test: 1968800
Total: 9997850


Expected: 9997850


In [14]:
# ==========================================================
# Every validation/test user should also exist in training
# ==========================================================

train_users = train_final.select("userId").distinct()

test_missing = (
    test.select("userId")
    .distinct()
    .subtract(train_users)
    .count()
)

val_missing = (
    val.select("userId")
    .distinct()
    .subtract(train_users)
    .count()
)

print(f"Test users missing from training: {test_missing}")
print(f"Validation users missing from training: {val_missing}")

Test users missing from training: 0
Validation users missing from training: 0


In [15]:
# ==========================================================
# Every validation/test movies should also exist in training
# ==========================================================

train_movies = train_final.select("movieId").distinct()

print(
    "Test movies missing from training: ",
    test.select("movieId").distinct().subtract(train_movies).count())
print(
    "Validation movies missing from training: ",
    val.select("movieId").distinct().subtract(train_movies).count())

Test movies missing from training:  2395


Validation movies missing from training:  1014


In [16]:
# ==========================================================
# Create evaluation users
# ==========================================================

evaluation_users = (
    test
    .select("userId")
    .distinct()
    .sample(False, 0.05, seed=52)
    .cache()
)

evaluation_users.count()

5185

### Baseline ALS Matrix Factorization
The baseline ALS model developed in Project 5 was reused as the collaborative filtering component of the hybrid recommender. Since the dataset, preprocessing steps, and data partitions remained unchanged, the optimal hyperparameters identified in Project 5 were retained instead of performing another hyperparameter search. The final model was trained using a rank of 40 latent factors, a regularization parameter of 0.01, and 20 training iterations. These settings previously demonstrated the best validation performance and provide the baseline recommendations that will be refined using director-based content information.

After training, the baseline model was evaluated on the independent test dataset to establish a performance benchmark for comparison with the proposed hybrid recommender. The model achieved a test RMSE of **0.8343** and MAE of **0.6151**, providing the baseline performance against which the hybrid recommender is evaluated.



In [17]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop", # Remove NaN predictions
    nonnegative=True, # Ensure that predicted ratings are non-negative
    implicitPrefs=False, # Explicit ratings
    seed=52
)

als_model = als.setParams(
    rank=40, # number of latent factors (hidden features)
    regParam=0.01, # regularization parameter, help prevent overfitting
    maxIter=20 # maximum number of ALS training iterations
).fit(train_final)

26/07/16 23:32:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [18]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Evaluation on the test set
als_test_predictions = als_model.transform(test)

als_test_rmse = evaluator.evaluate(als_test_predictions)

print(f"ALS Test RMSE = {als_test_rmse:.4f}")

ALS Test RMSE = 0.8335


In [19]:
# ==========================================================
# Generate Top-100 ALS recommendations directly
# ==========================================================

candidate_predictions = (
    als_model
    .recommendForUserSubset(evaluation_users, 100)
    .select(
        "userId",
        F.explode("recommendations").alias("rec")
    )
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("prediction")
    )
)

# print(candidate_predictions.count())

**Explanation:** This step generates a common candidate pool of up to 100 unseen movie recommendations for each evaluation user using the trained ALS model. The resulting candidate predictions serve as the input for both the baseline ALS recommender and the hybrid recommender, ensuring that both models are evaluated using the same candidate movies before the final Top-10 recommendations are produced.

### Hybrid Recommendation Model

To extend the baseline ALS recommender, a hybrid recommendation approach was implemented by incorporating movie director information into the recommendation process. While the ALS model captures latent relationships between users and movies based on historical ratings, it does not consider movie attributes that may influence user preferences. By integrating the **directedBy** metadata, the hybrid model incorporates content-based information alongside collaborative filtering predictions.

The hybrid recommendation process begins by generating candidate recommendations using the baseline ALS model. These candidate movies are then enriched with director information obtained from the movie metadata. For each user, directors associated with movies that received ratings of **3.5 or higher** in the training data are identified to represent the user's preferred directors, forming a director preference profile.

The director preferences are then incorporated into the ALS predictions using a weighted hybrid scoring approach. The final recommendation score is calculated as:
$$
\text{Final Score} = 0.8 \times \text{ALS Score} + 0.2 \times \text{Director Preference}
$$

where the Director Preference equals **1** if the recommended movie was directed by one of the user's preferred directors, and **0** otherwise.

This weighted approach allows collaborative filtering to remain the primary recommendation mechanism while incorporating director metadata to further personalize recommendation results. Movies are then re-ranked according to their final scores, and the highest-ranked unseen movies are presented as the final hybrid recommendations. These recommendations are subsequently evaluated against the baseline ALS model to assess the impact of incorporating director metadata.

In [21]:
# ==========================================================
# Create movie-level metadata
# ==========================================================

movie_metadata = (
    movie_ratings
    .select("movieId", "directedBy")
    .dropDuplicates(["movieId"])
    .cache()
)

# Materialize cache
movie_metadata.count()


# ==========================================================
# 1. Find movies each user liked
# ==========================================================

liked_movies = (
    train_final
    .filter(F.col("rating") >= 3.5)
)


# ==========================================================
# 2. Attach directors to liked movies
# ==========================================================

liked_directors = (
    liked_movies.alias("l")
    .join(
        movie_metadata.alias("m"),
        F.col("l.movieId") == F.col("m.movieId"),
        "left"
    )
    .select(
        F.col("l.userId").alias("userId"),
        F.col("l.movieId").alias("movieId"),
        F.col("m.directedBy").alias("directedBy")
    )
)


# ==========================================================
# 3. Create user favorite directors
# ==========================================================

favorite_directors = (
    liked_directors
    .filter(F.col("directedBy").isNotNull())
    .groupBy(
        "userId",
        "directedBy"
    )
    .count()
    .withColumnRenamed(
        "count",
        "director_count"
    )
    .select(
        "userId",
        "directedBy",
        "director_count"
    )
)


# ==========================================================
# 4. Add directors to ALS candidate predictions
# ==========================================================

als_with_directors = (
    candidate_predictions.alias("a")
    .join(
        movie_metadata.alias("m"),
        F.col("a.movieId") == F.col("m.movieId"),
        "left"
    )
    .select(
        F.col("a.userId").alias("userId"),
        F.col("a.movieId").alias("movieId"),
        F.col("a.prediction").alias("prediction"),
        F.col("m.directedBy").alias("directedBy")
    )
)


# ==========================================================
# 5. Combine ALS score with director preference
# ==========================================================

als_with_director_score = (
    als_with_directors.alias("als")
    .join(
        favorite_directors.alias("fav"),
        (
            (F.col("als.userId") == F.col("fav.userId")) &
            (F.col("als.directedBy") == F.col("fav.directedBy"))
        ),
        "left"
    )
    .select(
        F.col("als.userId").alias("userId"),
        F.col("als.movieId").alias("movieId"),
        F.col("als.prediction").alias("prediction"),
        F.col("fav.director_count").alias("director_count")
    )
    .withColumn(
        "directorPreference",
        F.when(
            F.col("director_count").isNotNull(),
            1
        )
        .otherwise(0)
    )
    .withColumn(
        "finalScore",
        (
            0.8 * F.col("prediction") +
            0.2 * F.col("directorPreference")
        )
    )
    .select(
        "userId",
        "movieId",
        "prediction",
        "finalScore"
    )
)


# ==========================================================
# 6. Rank hybrid recommendations and keep Top-10
# ==========================================================

window = Window.partitionBy("userId").orderBy(
    F.desc("finalScore")
)


final_hybrid_recommendations = (
    als_with_director_score
    .withColumn(
        "rank",
        F.row_number().over(window)
    )
    .filter(
        F.col("rank") <= 10
    )
    .drop("rank")
)


# Preview only
final_hybrid_recommendations.show(5)

26/07/16 23:35:58 WARN CacheManager: Asked to cache already cached data.


+------+-------+----------+------------------+
|userId|movieId|prediction|        finalScore|
+------+-------+----------+------------------+
|100007|   4163| 6.4421067| 5.153685379028321|
|100007| 150667|  5.619515| 4.495611953735351|
|100007| 182521|  5.598569| 4.478855133056641|
|100007| 142092|  5.318206| 4.254564666748047|
|100007| 175695| 5.1293764|4.1035011291503904|
+------+-------+----------+------------------+
only showing top 5 rows


**Explanation:** The hybrid recommender combines ALS collaborative filtering predictions with a simple content-based preference signal based on movie directors. First, each user's previously liked movies (ratings of 3.5 or higher) are used to identify their preferred directors. The ALS candidate recommendations are then enriched with director information and matched against each user's preferred directors. If a recommended movie is directed by one of the user's preferred directors, it receives a binary director preference score of 1; otherwise, it receives 0. The final hybrid score is calculated using a weighted combination of the ALS prediction (80%) and the director preference score (20%). This approach allows the model to retain ALS as the primary recommendation mechanism while incorporating director-based content information to further personalize the recommendation rankings. Finally, the recommendations are ranked according to the hybrid score, and the top 10 movies are selected for each user.

In [ ]:
# Display the number of recommendations per user in the final hybrid recommendations
final_hybrid_recommendations.groupBy("userId").count().describe().show()

+-------+------------------+-----+
|summary|            userId|count|
+-------+------------------+-----+
|  count|              5185| 5185|
|   mean|153598.72227579556| 10.0|
| stddev| 30939.10889165448|  0.0|
|    min|            100007|   10|
|    max|            206949|   10|
+-------+------------------+-----+



### Evaluation Framework

The performance of the recommendation systems was evaluated from two perspectives: prediction accuracy and recommendation quality. Prediction accuracy metrics were used to measure how well the models estimate user preferences, while ranking-based metrics were used to evaluate the relevance and quality of the generated recommendation lists.

RMSE and MAE were used to evaluate prediction accuracy. For the baseline ALS model, these metrics were calculated using the predicted ratings generated by the trained ALS model on the test dataset. For the hybrid recommender, RMSE and MAE were calculated using the final hybrid score, which combines the ALS prediction with the director preference score. Although these metrics provide an indication of how closely each model's scores align with the observed ratings, the hybrid scores are derived from a different scoring function and therefore should be interpreted separately from the baseline ALS prediction errors.

To ensure a fair comparison between the two recommendation approaches, both ALS and hybrid recommendations were evaluated using the same 1,000-user evaluation subset, the same Top-100 candidate recommendation pool generated by the ALS model, and identical Top-K ranking evaluation procedures. The ALS model was evaluated using its predicted ratings, while the hybrid model was evaluated using the combined hybrid score generated from ALS predictions and director preferences. This ensures that differences in recommendation quality are attributed to the incorporation of director metadata rather than differences in the evaluation setup.

Although RMSE and MAE provide insight into prediction accuracy, the primary objective of a recommendation system is to generate relevant ranked recommendations. Therefore, ranking-based evaluation metrics were considered the main criteria for assessing the effectiveness of the hybrid approach. Precision@10 and Recall@10 were used to measure recommendation relevance by evaluating whether highly rated movies from the test dataset appeared within the top-10 recommendations. In addition, Novelty@10 and Diversity@10 were included to measure recommendation quality beyond accuracy. Novelty evaluates whether the system recommends less popular movies rather than only highly popular ones, while Diversity measures the variety of directors represented within each user's recommendation list.

Due to computational limitations in the Databricks Free Tier environment, Top-K ranking evaluation was performed using a representative subset of users rather than the complete user population. A unified evaluation user set was created and applied consistently to both the baseline ALS model and the hybrid recommender. This approach reduces computational complexity while maintaining a fair comparison, as both models are evaluated under the same user conditions. Only users with available recommendation lists from both approaches were included to ensure that ranking metrics were calculated consistently across models.

The overall goal of the evaluation is to determine whether incorporating director metadata improves recommendation quality compared with the baseline ALS model. The hybrid model is expected to maintain competitive recommendation performance while improving personalization through better ranking quality and providing a balance between relevance, novelty, and diversity.




In [23]:
# Keep this
evaluation_users = (
    evaluation_users
    .join(
        final_hybrid_recommendations.select("userId").distinct(),
        "userId",
        "inner"
    )
)

**Explanation:** The evaluation user subset is filtered to retain only users who have recommendation lists generated by the hybrid recommender. This filtered user set is then used consistently when evaluating both the baseline ALS model and the hybrid model. Using the same evaluation users ensures that all Top-K metrics are calculated over an identical set of users, allowing a fair and consistent comparison between the two recommendation approaches.

In [24]:
# Create a clean movie metadata table
movies_meta = (
    movie_ratings
    .select("movieId", "title", "directedBy", "genres")
    .dropDuplicates(["movieId"])
)

# print("Movies metadata:", movies_meta.count())

## ALS Evaluation Metrics

In [25]:
als_top10 = (
    candidate_predictions
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("userId")
            .orderBy(F.desc("prediction"))
        )
    )
    .filter(F.col("rank") <= 10)
    .select("userId", "movieId")
    .cache()
)

als_top10.count()
# print("ALS Top10:", als_top10.count())

51850

In [26]:
def als_evaluate(model, train_df, test_df, movie_metadata, evaluation_users, k=10, threshold=3.5):

    # ==========================================================
    # 1. RMSE + MAE
    # ==========================================================

    preds = (model.transform(test_df)
        .dropna()
        )

    preds.count()

    rmse = (preds
        .select(((F.col("prediction") - F.col("rating")) ** 2)
            .alias("se"))
        .agg(F.sqrt(F.mean("se")).alias("RMSE"))
        .collect()[0][0])

    mae = (preds
        .select(F.abs(F.col("prediction") - F.col("rating"))
            .alias("ae")
        ).agg(F.mean("ae").alias("MAE"))
        .collect()[0][0])


    # ==========================================================
    # 2. TOP-K RECOMMENDATIONS
    # ==========================================================

    topk = als_top10


    # ==========================================================
    # 3. PRECISION + RECALL
    # ==========================================================

    relevant = (test_df
        .filter(F.col("rating") >= threshold)
        .select("userId", "movieId")
        .distinct())


    hits = topk.join(relevant,
        ["userId", "movieId"])


    user_hits = (hits
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "hits"))


    user_recs = (topk
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "recs"))


    user_rels = (relevant
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "rels"))


    stats = (user_recs
        .join(user_hits, "userId", "left")
        .join(user_rels, "userId", "left")
        .fillna(0))


    precision = (stats
        .select(F.mean(F.when(
                    F.col("recs") > 0,
                    F.col("hits") / F.col("recs"))
                .otherwise(0)))
        .collect()[0][0])


    recall = (stats
        .select(F.mean(F.when(
                    F.col("rels") > 0,
                    F.col("hits") / F.col("rels"))
                .otherwise(0)))
        .collect()[0][0])


    # ==========================================================
    # 4. NOVELTY
    # ==========================================================

    total_users = (train_df
        .select("userId")
        .distinct()
        .count())


    item_pop = (train_df
        .groupBy("movieId")
        .count()
        .withColumn("popularity",
            F.col("count") / F.lit(total_users)))


    novelty = (topk
        .join(item_pop.select("movieId", "popularity"),
            "movieId", "left")
        .fillna({"popularity": 1e-6})
        .withColumn("nov_score",
            -F.log2(F.col("popularity")))
        .agg(F.mean("nov_score"))
        .collect()[0][0])


    # ==========================================================
    # 5. DIVERSITY (Pandas)
    # ==========================================================

    diversity_df = (
        topk
        .join(
            movie_metadata.select("movieId", "directedBy"),
            "movieId",
            "left"
        )
        .select(
            "userId",
            "directedBy"
        )
        .toPandas()
    )


    def calculate_diversity(group):
        directors = group["directedBy"].tolist()

        if len(directors) < 2:
            return None

        total_pairs = 0
        different_pairs = 0

        for i in range(len(directors)):
            for j in range(i + 1, len(directors)):
                total_pairs += 1

                if directors[i] != directors[j]:
                    different_pairs += 1

        return different_pairs / total_pairs


    diversity = (
        diversity_df
        .groupby("userId")
        .apply(calculate_diversity)
        .mean()
    )


    return {
        "RMSE": rmse,
        "MAE": mae,
        "Precision@K": precision,
        "Recall@K": recall,
        "Novelty": novelty,
        "Diversity": diversity
    }

In [27]:
als_results = als_evaluate(
    als_model,
    train_final,
    test,
    movie_metadata,
    evaluation_users,
    k=10
)

In [28]:
als_results

{'RMSE': 0.8335269402404256,
 'MAE': 0.615136636827504,
 'Precision@K': 0.00027000964320154295,
 'Recall@K': 0.0023271634187857757,
 'Novelty': 14.763971011199068,
 'Diversity': np.float64(0.9721632915461267)}

## Hybrid Evaluation Metrics

In [ ]:
def hybrid_evaluate(
    recommendations,
    train_df,
    test_df,
    movies_df,
    evaluation_users,
    k=10,
    threshold=3.5
):

    # ==========================================================
    # Prepare Top-K recommendations
    # ==========================================================

    topk = (
        recommendations
        .select("userId", "movieId")
        .cache()
    )

    # Materialize cache
    topk.count()


    # ==========================================================
    # 1. RMSE + MAE
    # ==========================================================

    # Compare hybrid scores against actual ratings
    hybrid_predictions = (recommendations
        .join(test_df.select(
                "userId", "movieId",
                "rating"),
            ["userId", "movieId"], "inner")
    )


    hybrid_rmse = (hybrid_predictions
        .select(F.sqrt(F.mean(
                    (F.col("finalScore") - F.col("rating")) ** 2)
            ).alias("RMSE"))
        .collect()[0][0]
    )


    hybrid_mae = (
        hybrid_predictions
        .select(F.mean(F.abs(
                    F.col("finalScore") - F.col("rating"))
            ).alias("MAE"))
        .collect()[0][0])


    # ==========================================================
    # 2. Precision@K + Recall@K
    # ==========================================================

    relevant = (test_df
        .filter(F.col("rating") >= threshold)
        .select("userId", "movieId"
        ).distinct()
    )


    hits = (topk
        .join(relevant,
            ["userId", "movieId"], "inner")
    )


    user_hits = (hits
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "hits")
    )


    user_recs = (topk
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "recs")
    )


    user_rels = (relevant
        .groupBy("userId")
        .count()
        .withColumnRenamed("count", "rels")
    )


    stats = (user_recs
        .join(user_hits, "userId",
            "left")
        .join(user_rels, "userId",
            "left")
        .fillna(0)
    )


    precision = (stats
        .select(F.mean(F.when(
                    F.col("recs") > 0,
                    F.col("hits") / F.col("recs"))
                .otherwise(0)))
        .collect()[0][0]
    )


    recall = (stats
        .select(F.mean(F.when(
                    F.col("rels") > 0,
                    F.col("hits") / F.col("rels"))
                .otherwise(0)
            )
        )
        .collect()[0][0]
    )


    # ==========================================================
    # 3. Novelty
    # ==========================================================

    total_users = (
        train_df
        .select("userId")
        .distinct()
        .count()
    )


    item_pop = (
        train_df
        .groupBy("movieId")
        .count()
        .withColumn(
            "popularity",
            F.col("count") / F.lit(total_users))
    )


    novelty = (topk
        .join(item_pop.select("movieId", "popularity"),
            "movieId", "left")
        .fillna(
            {"popularity": 1e-6})
        .withColumn("nov_score", -F.log2(F.col("popularity")))
        .agg(F.mean("nov_score"))
        .collect()[0][0]
    )


    # ==========================================================
    # 4. Diversity
    # ==========================================================

    diversity_df = (topk
        .join(movies_df.select("movieId", "directedBy"),
            "movieId", "left")
        .select("userId", "directedBy")
        .toPandas()
    )


    def calculate_diversity(group):

        directors = (group["directedBy"]
            .dropna()
            .tolist()
        )

        if len(directors) < 2:
            return None

        total_pairs = 0
        different_pairs = 0

        for i in range(len(directors)):
            for j in range(i + 1, len(directors)):

                total_pairs += 1

                if directors[i] != directors[j]:
                    different_pairs += 1

        return different_pairs / total_pairs


    diversity = (
        diversity_df
        .groupby("userId")
        .apply(calculate_diversity)
        .mean()
    )


    # ==========================================================
    # Cleanup
    # ==========================================================

    topk.unpersist()


    return {
        "RMSE": hybrid_rmse,
        "MAE": hybrid_mae,
        "Precision@K": precision,
        "Recall@K": recall,
        "Novelty": novelty,
        "Diversity": diversity
    }

In [30]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [31]:
hybrid_results = hybrid_evaluate(
    final_hybrid_recommendations,
    train_final,
    test,
    movie_metadata,
    evaluation_users,
    k=10
)

print(hybrid_results)

{'RMSE': 0.897423020925034, 'MAE': 0.7466001415252681, 'Precision@K': 0.000540019286403086, 'Recall@K': 0.0032272047484176547, 'Novelty': 14.707080584734953, 'Diversity': np.float64(0.9711475409836064)}


In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "ALS Baseline",
        "Hybrid ALS + Director",
        "Hybrid ALS + Director"
    ],
    "ALS_Weight": [1.0, 0.6, 0.8],
    "Director_Weight": [0.0, 0.4, 0.2],
    "RMSE": [0.8335, 1.4476, 0.9142],
    "MAE": [0.6151, 1.3774, 0.7504],
    "Precision_at_K": [0.0003, 0.0012, 0.0005],
    "Recall_at_K": [0.0023, 0.0048, 0.0028],
    "Novelty": [14.7640, 14.5266, 14.7134],
    "Diversity": [0.9722, 0.9683, 0.9712]
})

display(comparison_df)

,Model,ALS_Weight,Director_Weight,RMSE,MAE,Precision_at_K,Recall_at_K,Novelty,Diversity
0,ALS Baseline,1.0,0.0,0.8335,0.6151,0.0003,0.0023,14.7640,0.9722
1,Hybrid ALS + Director,0.6,0.4,1.4476,1.3774,0.0012,0.0048,14.5266,0.9683
2,Hybrid ALS + Director,0.8,0.2,0.9142,0.7504,0.0005,0.0028,14.7134,0.9712


### Final Comparison

The effect of changing the contribution of director metadata was evaluated by comparing two hybrid weighting configurations. The first configuration assigned 80% weight to the ALS prediction and 20% weight to the director preference signal, while the second configuration increased the director contribution to 40%.

The 0.8/0.2 configuration achieved substantially lower RMSE (0.9142) and MAE (0.7504) than the 0.6/0.4 configuration (RMSE: 1.4476, MAE: 1.3774), indicating that its recommendation scores remained much closer to the observed user ratings. Although the 0.6/0.4 configuration produced higher Precision@10 (0.0012) and Recall@10 (0.0048), these improvements came at the cost of considerably worse prediction accuracy and slight reductions in both Novelty@10 and Diversity@10.

These results suggest that director metadata is most effective as a complementary recommendation signal rather than the primary ranking factor. Since director preference is represented as a binary feature (0 or 1), assigning it excessive weight causes it to influence the final recommendation score more strongly than the richer collaborative information learned by the ALS model. In contrast, the ALS predictions capture latent user-item relationships derived from historical rating patterns and therefore provide a more informative basis for ranking recommendations.

Considering the trade-off between prediction accuracy and recommendation quality, the 0.8 ALS / 0.2 director preference configuration was selected as the final hybrid model. This weighting maintains recommendation quality while preserving much better prediction accuracy than the alternative configuration, allowing director metadata to enhance personalization without overwhelming the collaborative filtering component.

## Conclusion

This project extended the baseline Alternating Least Squares (ALS) recommendation system by incorporating movie director metadata to develop a hybrid recommendation model. The hybrid approach combined collaborative filtering based on user rating behavior with content-based information derived from users' preferred directors, allowing the system to consider both user-item interactions and movie attributes when generating recommendations.

The evaluation framework compared the baseline ALS model and the hybrid recommender using both prediction accuracy and recommendation quality metrics. RMSE and MAE were used to evaluate prediction accuracy, while Precision@10, Recall@10, Novelty@10, and Diversity@10 were used to assess the quality of the generated recommendation lists. To ensure a fair comparison, both models were evaluated using the same evaluation user subset, the same Top-100 candidate recommendation pool generated by the ALS model, and an identical Top-K evaluation procedure.

The results showed that incorporating director metadata influenced the final recommendation rankings by introducing an additional content-based personalization signal. Increasing the contribution of director preference improved Precision@10 and Recall@10 but substantially reduced prediction accuracy, demonstrating the trade-off between recommendation relevance and rating prediction. The selected hybrid configuration, which assigned 80% weight to the ALS prediction and 20% weight to the director preference score, achieved a better overall balance by preserving much of the baseline model's prediction accuracy while incorporating personalized director preferences into the recommendation process.

Overall, the project demonstrates that movie director metadata can effectively complement collaborative filtering when incorporated with an appropriate weighting strategy. While the baseline ALS model remained superior in terms of prediction accuracy, the hybrid recommender illustrates how content-based information can be integrated to refine recommendation rankings and provide additional personalization. Future work could investigate incorporating richer movie metadata, such as genres, actors, writers, or plot descriptions, as well as exploring more advanced hybrid approaches that learn the contribution of collaborative and content-based features automatically.

### Implementation Note and Computational Limitations

The initial implementation was developed using the Databricks Community Edition environment, which was required for the project workflow and provided the Spark execution environment used for model development. However, limitations within the Databricks environment, including Unity Catalog restrictions and available computational resources, affected the ability to efficiently execute some recommendation and evaluation tasks.

Due to increasing computational requirements during the hybrid recommendation experiments, the workflow was subsequently migrated to a local Spark environment. While this allowed greater flexibility and control over the execution environment, the larger recommendation pipeline introduced significant memory and processing challenges. Multiple Spark jobs required optimization, including reducing intermediate data sizes, managing caching strategies, and adjusting recommendation generation procedures.

The development and evaluation process involved extensive experimentation due to repeated Spark execution failures caused by computational constraints. Several iterations of model generation, hybrid scoring, and evaluation were required, resulting in multiple days of experimentation to obtain stable results. Despite these limitations, the final implementation successfully generated comparable evaluation results for both the baseline ALS model and the hybrid recommender under the same evaluation methodology.

Additionally, Spark MLlib's built-in Top-N recommendation methods (recommendForAllUsers() and recommendForUserSubset()) could not be used in the original Databricks environment due to the UC_COMMAND_NOT_SUPPORTED.WITHOUT_RECOMMENDATION Unity Catalog limitation. Therefore, a manual recommendation pipeline was implemented using the learned user and item latent factors to generate candidate recommendations for evaluation.